# SkyHawk Roof Edge Detection — Training Notebook
Train a U-Net + ResNet-50 semantic segmentation model for roof edge detection from satellite imagery.

**Classes:** background (0), roof surface (1), ridge (2), hip (3), valley (4), eave/rake (5), flashing (6)

**Setup options (pick one):**
- **Option A (recommended):** Clone the GitHub repo directly into Colab (Cell 2A)
- **Option B:** Upload your `ml/` folder to Google Drive, mount Drive (Cell 2B)

Then run all remaining cells.

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -- switch to GPU runtime!'}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless tqdm PyYAML scikit-learn matplotlib onnx onnxruntime

# ============================================================
# OPTION A: Clone repo from GitHub (recommended)
# ============================================================
import os

REPO_URL = 'https://github.com/BrianPillmore/SkyHawk.git'
CLONE_DIR = '/content/SkyHawk'

if not os.path.exists(CLONE_DIR):
    print("Cloning SkyHawk repo...")
    os.system(f'git clone --depth 1 {REPO_URL} {CLONE_DIR}')
else:
    print("Repo already cloned, pulling latest...")
    os.system(f'cd {CLONE_DIR} && git pull')

ML_DIR = os.path.join(CLONE_DIR, 'ml')
print(f"ML_DIR set to: {ML_DIR}")

# ============================================================
# OPTION B: Use Google Drive (uncomment these lines instead)
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')
# ML_DIR = '/content/drive/MyDrive/SkyHawk/ml'

In [ ]:
import sys
import os
import glob

# Set up all paths from ML_DIR (defined in previous cell)
SCRIPTS_DIR = os.path.join(ML_DIR, 'scripts')
DATA_DIR = os.path.join(ML_DIR, 'data', 'annotated')
TRAIN_DIR = os.path.join(ML_DIR, 'data', 'train')
VAL_DIR = os.path.join(ML_DIR, 'data', 'val')
TEST_DIR = os.path.join(ML_DIR, 'data', 'test')
MODELS_DIR = os.path.join(ML_DIR, 'models')
CONFIG_PATH = os.path.join(ML_DIR, 'configs', 'model_config.yaml')

# Verify paths exist
assert os.path.exists(ML_DIR), f"ML_DIR not found: {ML_DIR}"
assert os.path.exists(SCRIPTS_DIR), f"Scripts not found: {SCRIPTS_DIR}"
assert os.path.exists(CONFIG_PATH), f"Config not found: {CONFIG_PATH}"

print(f"ML_DIR:     {ML_DIR}")
print(f"Scripts:    {sorted(os.listdir(SCRIPTS_DIR))}")
print(f"Config:     {CONFIG_PATH}")

# Add scripts to Python path
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

# Create output dirs
for d in [MODELS_DIR, DATA_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(d, exist_ok=True)

# Check for annotated data
pairs = [f for f in glob.glob(os.path.join(DATA_DIR, '*.png')) if '_mask' not in f]
masks = glob.glob(os.path.join(DATA_DIR, '*_mask.png'))
print(f"\nAnnotated data: {len(pairs)} images, {len(masks)} masks")

if len(pairs) == 0:
    print("\nNo annotated images found yet.")
    print(f"Upload image+mask pairs to: {DATA_DIR}/")
    print("Format: <name>.png + <name>_mask.png")
    print("\nYou can upload files in Colab using the file browser (folder icon on the left)")
    print("or use: from google.colab import files; files.upload()")

In [ ]:
# === Upload annotated images ===
# Run this cell to upload image+mask pairs from your local machine.
# Skip if you already have data in the annotated directory.

from google.colab import files
import shutil

print("Upload your annotated image+mask pairs.")
print("Each image needs two files: <name>.png and <name>_mask.png")
print("(You can select multiple files at once)\n")

uploaded = files.upload()

for filename, data in uploaded.items():
    dest = os.path.join(DATA_DIR, filename)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"  Saved: {dest}")

# Recount
pairs = [f for f in glob.glob(os.path.join(DATA_DIR, '*.png')) if '_mask' not in f]
masks = glob.glob(os.path.join(DATA_DIR, '*_mask.png'))
print(f"\nTotal: {len(pairs)} images, {len(masks)} masks")

In [ ]:
# Ensure scripts path is set (in case cells are run out of order)
import sys, os, glob
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

import yaml
import importlib
import dataset as dataset_mod
import model as model_mod
importlib.reload(dataset_mod)
importlib.reload(model_mod)
from dataset import split_dataset

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

print("Config loaded:")
print(f"  Encoder: {config['model']['encoder']}")
print(f"  Classes: {config['model']['classes']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Learning rate: {config['training']['learning_rate']}")

# Split into train/val/test if not already done
if not os.path.exists(TRAIN_DIR) or len(glob.glob(os.path.join(TRAIN_DIR, '*.png'))) == 0:
    print("\nSplitting data into train/val/test...")
    split_dataset(DATA_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR)
else:
    train_count = len([f for f in glob.glob(os.path.join(TRAIN_DIR, '*.png')) if '_mask' not in f])
    val_count = len([f for f in glob.glob(os.path.join(VAL_DIR, '*.png')) if '_mask' not in f])
    test_count = len([f for f in glob.glob(os.path.join(TEST_DIR, '*.png')) if '_mask' not in f])
    print(f"\nExisting split: train={train_count}, val={val_count}, test={test_count}")

In [ ]:
# Ensure scripts path is set
import sys
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

from model import create_model, CombinedLoss
from dataset import RoofEdgeDataset, get_train_augmentations, get_val_augmentations

# Model
model = create_model(config).cuda()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: U-Net + {config['model']['encoder']}")
print(f"Parameters: {total_params:,}")

# Datasets
train_aug = get_train_augmentations(config)
val_aug = get_val_augmentations()

train_dataset = RoofEdgeDataset(TRAIN_DIR, transform=train_aug)
val_dataset = RoofEdgeDataset(VAL_DIR, transform=val_aug)

print(f"Training:   {len(train_dataset)} images")
print(f"Validation: {len(val_dataset)} images")

# Quick sanity check
sample = train_dataset[0]
print(f"\nSample image shape: {sample['image'].shape}")
print(f"Sample mask shape:  {sample['mask'].shape}")
print(f"Mask classes present: {torch.unique(sample['mask']).tolist()}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CLASS_NAMES = ['bg', 'roof', 'ridge', 'hip', 'valley', 'eave', 'flash']
CLASS_COLORS = np.array([
    [0, 0, 0], [128, 128, 128], [255, 0, 0], [255, 165, 0],
    [0, 0, 255], [0, 255, 0], [192, 192, 192]
]) / 255.0

def show_sample(dataset, idx=0):
    sample = dataset[idx]
    img = sample['image'].numpy().transpose(1, 2, 0)
    # Denormalize
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = np.clip(img * std + mean, 0, 1)
    
    mask = sample['mask'].numpy()
    mask_color = CLASS_COLORS[mask]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img)
    axes[0].set_title(f"Image: {sample['filename']}")
    axes[1].imshow(mask_color)
    axes[1].set_title("Ground Truth Mask")
    axes[2].imshow(img)
    axes[2].imshow(mask_color, alpha=0.4)
    axes[2].set_title("Overlay")
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_sample(train_dataset, 0)

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm
import time, json

# Hyperparameters
train_cfg = config['training']
EPOCHS = train_cfg['epochs']
BATCH_SIZE = train_cfg['batch_size']
LR = train_cfg['learning_rate']
PATIENCE = train_cfg['early_stopping_patience']

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Loss
criterion = CombinedLoss(
    class_weights=train_cfg['class_weights'],
    ce_weight=train_cfg['loss']['ce_weight'],
    dice_weight=train_cfg['loss']['dice_weight'],
).cuda()

# Optimizer + Scheduler
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=train_cfg['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

def compute_iou(pred, target, num_classes=7):
    ious = {}
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    for c in range(num_classes):
        pred_c = (pred_flat == c)
        target_c = (target_flat == c)
        inter = (pred_c & target_c).sum().item()
        union = (pred_c | target_c).sum().item()
        ious[c] = inter / union if union > 0 else float('nan')
    edge_ious = [ious[c] for c in range(2, 7) if not np.isnan(ious[c])]
    ious['edge_mean'] = np.mean(edge_ious) if edge_ious else 0.0
    valid = [v for v in ious.values() if isinstance(v, float) and not np.isnan(v)]
    ious['mean'] = np.mean(valid) if valid else 0.0
    return ious

# Training
history = []
best_edge_iou = 0.0
no_improve = 0

print(f"Training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}, patience={PATIENCE}")
print(f"{'='*70}")

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Train", leave=False):
        images = batch['image'].cuda()
        masks = batch['mask'].cuda()
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_dataset)
    
    # Validate
    model.eval()
    val_loss = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Val", leave=False):
            images = batch['image'].cuda()
            masks = batch['mask'].cuda()
            logits = model(images)
            val_loss += criterion(logits, masks).item() * images.size(0)
            all_preds.append(logits.argmax(dim=1).cpu())
            all_targets.append(masks.cpu())
    val_loss /= len(val_dataset)
    
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    ious = compute_iou(all_preds, all_targets)
    edge_iou = ious['edge_mean']
    
    scheduler.step()
    elapsed = time.time() - t0
    
    print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
          f"mIoU: {ious['mean']:.4f} | Edge mIoU: {edge_iou:.4f} | {elapsed:.1f}s")
    
    history.append({
        'epoch': epoch+1, 'train_loss': train_loss, 'val_loss': val_loss,
        'mean_iou': ious['mean'], 'edge_mean_iou': edge_iou,
    })
    
    # Save best
    if edge_iou > best_edge_iou:
        best_edge_iou = edge_iou
        no_improve = 0
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_edge_iou': best_edge_iou, 'config': config,
        }, os.path.join(MODELS_DIR, 'best.pth'))
        print(f"  \u2713 New best: {best_edge_iou:.4f}")
    else:
        no_improve += 1
    
    # Save latest
    torch.save({
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_edge_iou': best_edge_iou, 'config': config,
    }, os.path.join(MODELS_DIR, 'latest.pth'))
    
    if no_improve >= PATIENCE:
        print(f"\nEarly stopping after {PATIENCE} epochs without improvement")
        break

# Save history
with open(os.path.join(MODELS_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

print(f"\n{'='*70}")
print(f"Done! Best edge mIoU: {best_edge_iou:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_list = [h['epoch'] for h in history]

axes[0].plot(epochs_list, [h['train_loss'] for h in history], label='Train')
axes[0].plot(epochs_list, [h['val_loss'] for h in history], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_list, [h['mean_iou'] for h in history], label='Mean IoU')
axes[1].plot(epochs_list, [h['edge_mean_iou'] for h in history], label='Edge Mean IoU')
axes[1].set_title('IoU')
axes[1].legend()
axes[1].set_xlabel('Epoch')
axes[1].axhline(y=0.30, color='r', linestyle='--', alpha=0.5, label='Initial target')
axes[1].axhline(y=0.50, color='orange', linestyle='--', alpha=0.5, label='Usable target')

axes[2].plot(epochs_list, [h['edge_mean_iou'] for h in history], 'g-', linewidth=2)
axes[2].set_title('Edge Mean IoU (primary metric)')
axes[2].set_xlabel('Epoch')
axes[2].axhline(y=0.30, color='r', linestyle='--', alpha=0.3)
axes[2].axhline(y=0.50, color='orange', linestyle='--', alpha=0.3)
axes[2].axhline(y=0.65, color='green', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Ensure scripts path is set
import sys
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

# Load best model
ckpt = torch.load(os.path.join(MODELS_DIR, 'best.pth'))
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best model (epoch {ckpt['epoch']+1}, edge IoU {ckpt['best_edge_iou']:.4f})")

# Evaluate on test set
from dataset import RoofEdgeDataset, get_val_augmentations
test_dataset = RoofEdgeDataset(TEST_DIR, transform=get_val_augmentations())
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

all_preds, all_targets, all_images, all_names = [], [], [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        images = batch['image'].cuda()
        preds = model(images).argmax(dim=1).cpu()
        all_preds.append(preds)
        all_targets.append(batch['mask'])
        all_images.append(batch['image'])
        all_names.extend(batch['filename'])

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)
all_images = torch.cat(all_images)

test_ious = compute_iou(all_preds, all_targets)
print(f"\nTest Results:")
for c in range(7):
    iou = test_ious[c]
    marker = ' <- edge' if c >= 2 else ''
    if not np.isnan(iou):
        print(f"  {CLASS_NAMES[c]:8s}: {iou:.4f}{marker}")
    else:
        print(f"  {CLASS_NAMES[c]:8s}: N/A")
print(f"\n  Mean IoU:      {test_ious['mean']:.4f}")
print(f"  Edge Mean IoU: {test_ious['edge_mean']:.4f}")

# Visual overlays
n_show = min(8, len(all_images))
fig, axes = plt.subplots(n_show, 3, figsize=(15, 5 * n_show))
if n_show == 1:
    axes = axes.reshape(1, -1)

mean_np = np.array([0.485, 0.456, 0.406])
std_np = np.array([0.229, 0.224, 0.225])

for i in range(n_show):
    img = all_images[i].numpy().transpose(1, 2, 0)
    img = np.clip(img * std_np + mean_np, 0, 1)
    gt = CLASS_COLORS[all_targets[i].numpy()]
    pred = CLASS_COLORS[all_preds[i].numpy()]
    
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Image: {all_names[i]}")
    axes[i, 1].imshow(gt)
    axes[i, 1].set_title("Ground Truth")
    axes[i, 2].imshow(pred)
    axes[i, 2].set_title("Prediction")
    for j in range(3):
        axes[i, j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Export best model to ONNX
model.cpu()
model.eval()

dummy_input = torch.randn(1, 3, 640, 640)
onnx_path = os.path.join(MODELS_DIR, 'roof_edge_detector.onnx')

torch.onnx.export(
    model, dummy_input, onnx_path,
    input_names=['image'], output_names=['segmentation'],
    opset_version=17,
    dynamic_axes={'image': {0: 'batch_size'}, 'segmentation': {0: 'batch_size'}},
)

size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"ONNX model exported: {onnx_path} ({size_mb:.1f} MB)")

# Validate
import onnxruntime as ort
session = ort.InferenceSession(onnx_path)
onnx_out = session.run(None, {'image': dummy_input.numpy()})[0]
torch_out = model(dummy_input).detach().numpy()
max_diff = np.abs(torch_out - onnx_out).max()
status = "PASS" if max_diff < 1e-4 else "CHECK"
print(f"ONNX validation: max diff = {max_diff:.2e} {status}")

# Update versions.json
import json
versions_path = os.path.join(MODELS_DIR, 'versions.json')
versions = {'current': 'v1', 'versions': {
    'v1': {
        'trainedAt': time.strftime('%Y-%m-%d'),
        'images': len(train_dataset) + len(val_dataset) + len(test_dataset),
        'edgeIoU': float(best_edge_iou),
        'path': 'roof_edge_detector.onnx',
    }
}}
with open(versions_path, 'w') as f:
    json.dump(versions, f, indent=2)
print(f"versions.json updated")
print(f"\nDone! Copy {onnx_path} to your server's ml/models/ folder")
print(f"Then install: npm install onnxruntime-node sharp")

## Next Steps

1. **Download** the `roof_edge_detector.onnx` file from Drive
2. **Copy** it to `ml/models/` in your SkyHawk project
3. **Install** server dependencies: `npm install onnxruntime-node sharp`
4. **Restart** the server — the model auto-loads on first inference
5. **Test** via auto-detect on any property — ML model will be used automatically

### Improving the Model
- Annotate more images (target: 200+ for usable quality)
- Use the CorrectionOverlay in the app to fix detection errors
- Re-run this notebook with the expanded dataset
- The active learning loop (`ml/scripts/retrain.py`) can also fine-tune from corrections